# `StructurePreprocessor.encode_domains()`

`encode_domains` turns a domain segmentation into a `[0, 1]`-normalized `dict_num` with per-residue channels: `domain_boundary` (1 at a domain start or end), `domain_relative_position` (0 to 1 along the residue's domain), `domain_size` (domain length over protein length) and `n_domains_in_protein`. It is bring-your-own-segmentation: the **chopping string** (Merizo / ChainSaw / AFragmenter format, domains separated by commas, discontinuous segments joined by underscores, 1-based inclusive ranges) comes either from one file per entry in `domain_folder` (`<entry>.txt` or a `.tsv` with a `chopping` header) or from a `chopping` column that `get_domains` adds to `df_seq`.

In [1]:
import warnings
import tempfile
from pathlib import Path
import numpy as np
import aaanalysis as aa
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

df_seq = aa.load_dataset(name='DOM_GSEC', n=10)
df_seq = df_seq[df_seq['entry'].isin(['Q14802', 'O43914', 'P01135'])].reset_index(drop=True)
strp = aa.StructurePreprocessor(verbose=False)

# One chopping file per entry (here hand-written; in practice the output of Merizo,
# ChainSaw or get_domains). O43914 carries a discontinuous first domain.
chopping = {'Q14802': '1-30,37-87', 'O43914': '1-41_65-113,42-64', 'P01135': '24-98,99-160'}
domain_dir = Path(tempfile.mkdtemp()) / 'domains'
domain_dir.mkdir()
for entry, chop in chopping.items():
    (domain_dir / f'{entry}.txt').write_text(chop + '\n')

features = ['domain_boundary', 'domain_relative_position', 'domain_size', 'n_domains_in_protein']
dict_dom = strp.encode_domains(df_seq=df_seq, domain_folder=domain_dir, features=features)
print({entry: arr.shape for entry, arr in dict_dom.items()})
print('first residues of Q14802 (boundary, relative position, size, n_domains):')
print(np.round(dict_dom['Q14802'][:4], 2))

{'Q14802': (87, 4), 'P01135': (160, 4), 'O43914': (113, 4)}
first residues of Q14802 (boundary, relative position, size, n_domains):
[[1.   0.   0.15 0.2 ]
 [0.   0.03 0.15 0.2 ]
 [0.   0.07 0.15 0.2 ]
 [0.   0.1  0.15 0.2 ]]


When `df_seq` already carries a `chopping` column (the output of `get_domains`), no folder is needed:

In [2]:
df_seq_chop = df_seq.copy()
df_seq_chop['chopping'] = [chopping[entry] for entry in df_seq_chop['entry']]
dict_dom_col = strp.encode_domains(df_seq=df_seq_chop, features=['domain_size'])
print({entry: arr.shape for entry, arr in dict_dom_col.items()})

{'Q14802': (87, 1), 'P01135': (160, 1), 'O43914': (113, 1)}


## Further parameters

`on_failure` decides what happens to entries whose chopping file is missing or unparsable (`'nan'` fills a NaN tensor, `'drop'` removes them, `'raise'` raises); `return_df=True` also returns the per-row status frame with a `domain_ok` column. Below, one chopping file is removed first so the failure policy is visible.

In [3]:
(domain_dir / 'P01135.txt').unlink()
dict_dom_kept, df_status = strp.encode_domains(df_seq=df_seq, domain_folder=domain_dir,
                                               features=['domain_boundary'], on_failure='drop', return_df=True)
print('kept entries:', list(dict_dom_kept))
aa.display_df(df_status[['entry', 'gene', 'domain_ok']], n_rows=10, show_shape=True)

kept entries: ['Q14802', 'O43914']
DataFrame shape: (2, 3)


,entry,gene,domain_ok
1,Q14802,FXYD3,True
2,O43914,TYROBP,True
